# CosMx–MERFISH functional annotation and module analysis

This notebook reproduces the filtered functional annotation and cross-study
module analysis. By default, it uses frozen annotation inputs generated locally with the separate
`recover_filtered_mygene_annotations.py` script rather than querying the current
MyGeneInfo database.

The original analysis retained annotations associated with at least five genes
within a cell type. The same threshold and downstream score calculations are
preserved here.

## 0. Setup

In [1]:
import json
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import spearmanr, pearsonr

from scale_aware_st import RepositoryConfig

config = RepositoryConfig.from_env()
main_dir = config.results_dir / "cosmx"
resource_dir = config.resources_dir / "functional_annotations"
main_dir.mkdir(parents=True, exist_ok=True)

cosmx_adata_file = config.data_dir / "cosmx" / "cosmx_data_ct_final.h5ad"
annotation_file = resource_dir / "filtered_mygene_annotations_crossstudy.xlsx"

old_merfish_module_file = resource_dir / "module_dictionary.json"
cosmx_module_file = resource_dir / "cosmx_module_dictionary.json"
updated_merfish_module_file = resource_dir / "updated_merfish_module_dictionary.json"
shared_module_file = resource_dir / "shared_crossstudy_module_dictionary.json"

RUN_MYGENE = False
PLOT_EXAMPLE_GENES = True

MIN_GENES_PER_TERM = 5

out_cosmx_terms = main_dir / "FILT_cosmx_go_kegg_terms_for_module.xlsx"
out_merfish_terms = main_dir / "FILT_maindata_go_kegg_terms_for_module.xlsx"
out_cosmx_scores = main_dir / "cosmx_module_scores.xlsx"
out_merfish_scores = main_dir / "merfish_module_scores.xlsx"

## 1. Representative CosMx expression plots

In [ ]:
if PLOT_EXAMPLE_GENES:
    cosmx_mapped = sc.read_h5ad(cosmx_adata_file)
    cosmx_mapped.X = cosmx_mapped.layers["raw_counts"]

    cosmx_glia_ss = cosmx_mapped[
        cosmx_mapped.obs["final_class_final"].isin(
            ["Astro", "OLG", "Immune", "EC", "Ependymal"]
        )
    ].copy()

    ylims = {
        "Ywhaz": 150,
        "Mc2r": 8,
        "Aspa": 15,
        "Apoe": 20,
        "Adcy2": 20,
        "Adcy9": 40,
        "Htr2c": 25,
        "Dync1h1": 20,
    }

    genes = [
        "Apoe", "Adcy9", "Adcy2", "Dync1h1",
        "Htr2c", "Aspa", "Mc2r", "Ywhaz",
    ]

    fig, axes = plt.subplots(
        nrows=4,
        ncols=2,
        figsize=(14, 16),
        sharex=True,
    )
    axes = axes.flatten()

    for i, gene in enumerate(genes):
        plot_df = pd.DataFrame(
            {
                gene: cosmx_glia_ss[:, gene]
                .layers["raw_counts"]
                .flatten(),
                "cell_type": cosmx_glia_ss.obs[
                    "final_class_final"
                ].values,
                "age": cosmx_glia_ss.obs["age"].values,
            }
        )

        ax = axes[i]

        sns.violinplot(
            data=plot_df,
            x="cell_type",
            y=gene,
            hue="age",
            split=True,
            inner="quart",
            cut=0,
            width=1,
            linewidth=1,
            ax=ax,
        )

        ax.set_ylabel(gene, fontsize=16)
        ax.tick_params(axis="y", labelsize=14)
        ax.set_ylim(0, ylims[gene])

        if i == 0:
            ax.legend(
                title="Age",
                fontsize=14,
                title_fontsize=16,
            )
        else:
            ax.get_legend().remove()

        if i < 6:
            ax.set_xlabel("")
            ax.tick_params(axis="x", labelbottom=False)
        else:
            ax.set_xlabel("Cell Type", fontsize=14)
            ax.tick_params(axis="x", labelsize=12, rotation=60)

    plt.tight_layout()
    plt.show()

## 2. Load frozen annotations or optionally re-query MyGeneInfo

The default path loads the filtered, frozen annotation workbook reconstructed
from the original analysis. Setting `RUN_MYGENE = True` re-queries the genes
present in that workbook. Because MyGeneInfo, GO, and KEGG are updated over
time, live results are not expected to exactly reproduce the frozen analysis.

In [ ]:
def split_terms(value):
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [
        term.strip()
        for term in str(value).split(";")
        if term.strip()
    ]


def load_frozen_annotation_sheet(path, sheet_name):
    return pd.read_excel(path, sheet_name=sheet_name)


def as_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]


def query_gene_mygene(gene, mg):
    result = {
        "go_terms": [],
        "kegg_paths": [],
    }

    response = mg.query(
        gene,
        species="mouse",
        fields="go,pathway",
    )
    hits = response.get("hits", [])
    if not hits:
        return result

    hit = hits[0]

    go = hit.get("go", {})
    if isinstance(go, dict) and "BP" in go:
        result["go_terms"] = sorted(
            {
                entry.get("term")
                for entry in as_list(go["BP"])
                if isinstance(entry, dict)
                and entry.get("term")
            }
        )

    pathway = hit.get("pathway", {})
    if isinstance(pathway, dict) and "kegg" in pathway:
        result["kegg_paths"] = sorted(
            {
                entry.get("name")
                for entry in as_list(pathway["kegg"])
                if isinstance(entry, dict)
                and entry.get("name")
            }
        )

    return result


def requery_annotation_sheet(frozen_df):
    try:
        import mygene
    except ImportError as exc:
        raise ImportError(
            'RUN_MYGENE=True requires the optional mygene package.'
        ) from exc

    mg = mygene.MyGeneInfo()
    queried = []

    for (cell_type, gene), _ in frozen_df.groupby(
        ["cell_type", "gene"]
    ):
        info = query_gene_mygene(gene, mg)
        queried.append(
            {
                "cell_type": cell_type,
                "gene": gene,
                "go_terms": "; ".join(info["go_terms"]),
                "kegg_paths": "; ".join(info["kegg_paths"]),
            }
        )

    return pd.DataFrame(queried)


cosmx_annotations = load_frozen_annotation_sheet(
    annotation_file,
    "CosMx",
)
merfish_annotations = load_frozen_annotation_sheet(
    annotation_file,
    "MERFISH",
)

if RUN_MYGENE:
    cosmx_annotations = requery_annotation_sheet(
        cosmx_annotations
    )
    merfish_annotations = requery_annotation_sheet(
        merfish_annotations
    )

## 3. Reconstruct cell-type annotation count tables

In [ ]:
def build_annotation_counts(annotation_df):
    rows = []

    for _, row in annotation_df.iterrows():
        for annotation_type, column in [
            ("GO_Term", "go_terms"),
            ("KEGG_Path", "kegg_paths"),
        ]:
            for term in split_terms(row[column]):
                rows.append(
                    {
                        "cell_type": row["cell_type"],
                        "gene": row["gene"],
                        "Functional_Annotation": annotation_type,
                        "category": term,
                    }
                )

    long_df = pd.DataFrame(rows).drop_duplicates()

    count_rows = []
    for (
        cell_type,
        annotation_type,
        category,
    ), sub in long_df.groupby(
        [
            "cell_type",
            "Functional_Annotation",
            "category",
        ]
    ):
        genes = sorted(sub["gene"].unique())

        count_rows.append(
            {
                "cell_type": cell_type,
                "Functional_Annotation": annotation_type,
                "category": category,
                "n_occurrences": len(genes),
                "unique_genes": ", ".join(genes),
                "n_unique_genes": len(genes),
            }
        )

    counts = (
        pd.DataFrame(count_rows)
        .sort_values(
            [
                "cell_type",
                "Functional_Annotation",
                "n_occurrences",
            ],
            ascending=[True, True, False],
        )
        .reset_index(drop=True)
    )

    return counts


cosmx_counts = build_annotation_counts(cosmx_annotations)
maindata_counts = build_annotation_counts(merfish_annotations)

cosmx_counts = cosmx_counts.loc[
    cosmx_counts["n_occurrences"] >= MIN_GENES_PER_TERM
].copy()

maindata_counts = maindata_counts.loc[
    maindata_counts["n_occurrences"] >= MIN_GENES_PER_TERM
].copy()

## 4. Restrict both datasets to shared modules and terms

In [ ]:
with open(shared_module_file, "r", encoding="utf-8") as f:
    shared_modules = json.load(f)["modules"]

published_module_labels = {
    'Addiction_Substance_Response_Wrapper': 'Addiction Substance Response Wrapper',
    'Cancer_Wrapper_Pathways': 'Cancer Wrapper Pathways',
    'ECM_Adhesion_Cytoskeleton_Remodeling': 'ECM/ Adhesion/ Cytoskeleton Remodeling',
    'Energy_Metabolism_Mitochondria': 'Energy/ Metabolism/ Mitochondria',
    'Growth_Survival_Stress_Signaling': 'Growth/ Survival/ Stress Signaling',
    'Immune_Inflammation_Stress_Response': 'Immune/ Inflammation/ Stress Response',
    'Neurodegeneration_Disease_Wrapper': 'Neurodegeneration Disease Wrapper',
    'Neuronal_Synaptic_Plasticity_Programs': 'Neuronal Synaptic Plasticity Programs',
    'Proteostasis_DNA_Repair_Quality_Control': 'Proteostasis/ DNA Repair/ Quality Control',
    'Transport_Trafficking_Membrane_Homeostasis': 'Transport/ Trafficking/ Membrane Homeostasis',
}
missing_module_labels = set(shared_modules).difference(published_module_labels)
if missing_module_labels:
    raise KeyError(f'Missing published labels for modules: {sorted(missing_module_labels)}')

term_to_module = {
    term: published_module_labels[module]
    for module, terms in shared_modules.items()
    for term in terms
}

cosmx_counts_gk = cosmx_counts.copy()
cosmx_counts_gk["Module"] = (
    cosmx_counts_gk["category"].map(term_to_module)
)
cosmx_counts_gk = cosmx_counts_gk.loc[
    cosmx_counts_gk["Module"].notna()
].copy()

maindata_counts_gk = maindata_counts.copy()
maindata_counts_gk["Module"] = (
    maindata_counts_gk["category"].map(term_to_module)
)
maindata_counts_gk = maindata_counts_gk.loc[
    maindata_counts_gk["Module"].notna()
].copy()

cosmx_counts_gk.to_excel(out_cosmx_terms, index=False)
maindata_counts_gk.to_excel(out_merfish_terms, index=False)

## 5. Gene-term, term, and module score calculations

In [ ]:
def calculate_scores(counts_df):
    gene_term_df = counts_df.copy()

    gene_term_df["gene"] = (
        gene_term_df["unique_genes"].str.split(",")
    )
    gene_term_df = gene_term_df.explode("gene")
    gene_term_df["gene"] = gene_term_df["gene"].str.strip()

    gene_term_df["n_terms_gene"] = (
        gene_term_df.groupby(
            ["cell_type", "gene"]
        )["category"]
        .transform("nunique")
    )

    gene_term_df["gene_term_score"] = (
        1 / gene_term_df["n_terms_gene"]
    )

    term_scores = (
        gene_term_df.groupby(
            ["cell_type", "category", "Module"]
        )["gene_term_score"]
        .sum()
        .reset_index()
    )

    module_scores = (
        gene_term_df.groupby(
            ["cell_type", "Module"]
        )["gene_term_score"]
        .sum()
        .reset_index()
    )

    # Preserve original calculation: number of terms is
    # module-wide across the dataset, not cell-type-specific.
    module_scores = module_scores.merge(
        term_scores.groupby("Module")["category"]
        .nunique()
        .rename("n_terms"),
        on="Module",
    )

    module_scores["score_per_term"] = (
        module_scores["gene_term_score"]
        / module_scores["n_terms"]
    )

    module_scores["weighted_score"] = (
        module_scores["gene_term_score"]
        / module_scores["n_terms"] ** 0.5
    )

    module_scores["weighted_score_norm"] = (
        module_scores["weighted_score"]
        / module_scores.groupby("cell_type")[
            "weighted_score"
        ].transform("sum")
    )

    return gene_term_df, term_scores, module_scores


cosmx_gene_term_scores, cosmx_term_scores, cosmx_module_scores = (
    calculate_scores(cosmx_counts_gk)
)

merfish_gene_term_scores, merfish_term_scores, merfish_module_scores = (
    calculate_scores(maindata_counts_gk)
)

cosmx_module_scores.to_excel(
    out_cosmx_scores,
    index=False,
)
merfish_module_scores.to_excel(
    out_merfish_scores,
    index=False,
)

## 6. Module-level cross-study comparison (Fig 6D)

In [ ]:
score_col = "weighted_score_norm"

df_plot = (
    merfish_module_scores[
        ["cell_type", "Module", score_col]
    ]
    .merge(
        cosmx_module_scores[
            ["cell_type", "Module", score_col]
        ],
        on=["cell_type", "Module"],
        suffixes=("_merfish", "_cosmx"),
    )
)

cell_types = sorted(df_plot["cell_type"].unique())
module_order = sorted(df_plot["Module"].unique())

palette = dict(
    zip(
        module_order,
        sns.color_palette(
            "tab10",
            n_colors=len(module_order),
        ),
    )
)

fig, axes = plt.subplots(
    1,
    len(cell_types),
    figsize=(4 * len(cell_types), 4),
    sharex=False,
    sharey=False,
)

if len(cell_types) == 1:
    axes = [axes]

handles = labels = None

for ax, cell_type in zip(axes, cell_types):
    sub = df_plot.loc[
        df_plot["cell_type"] == cell_type
    ]

    sns.scatterplot(
        data=sub,
        x=f"{score_col}_merfish",
        y=f"{score_col}_cosmx",
        hue="Module",
        hue_order=module_order,
        palette=palette,
        s=150,
        ax=ax,
    )

    if handles is None:
        handles, labels = ax.get_legend_handles_labels()

    legend = ax.get_legend()
    if legend is not None:
        legend.remove()

    minimum = min(
        sub[f"{score_col}_merfish"].min(),
        sub[f"{score_col}_cosmx"].min(),
    )
    maximum = max(
        sub[f"{score_col}_merfish"].max(),
        sub[f"{score_col}_cosmx"].max(),
    )

    ax.plot(
        [minimum, maximum],
        [minimum, maximum],
        "--",
        color="black",
        alpha=0.5,
    )

    spearman_r, _ = spearmanr(
        sub[f"{score_col}_merfish"],
        sub[f"{score_col}_cosmx"],
    )
    pearson_r, _ = pearsonr(
        sub[f"{score_col}_merfish"],
        sub[f"{score_col}_cosmx"],
    )

    ax.set_title(
        f"{cell_type}\nρ={spearman_r:.2f}, "
        f"r={pearson_r:.2f}"
    )
    ax.set_xlabel("MERFISH")
    ax.set_ylabel("CosMx")

fig.legend(
    handles,
    labels,
    title="Module",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
)

plt.tight_layout(rect=[0, 0, 0.85, 1])
plt.show()